In [46]:
import pandas as pd
import requests
import json
import tabulate

In [28]:
CSV_PATH = "data/data.csv" 

df = pd.read_csv(CSV_PATH)
print("Nombres de lignes :", len(df))
display(df.head())


Nombres de lignes : 6894


,ID,FoodGroup,Descrip,Energy_kcal,Protein_g,Fat_g,Carb_g,Sugar_g,Fiber_g,VitA_mcg,...,Folate_USRDA,Niacin_USRDA,Riboflavin_USRDA,Thiamin_USRDA,Calcium_USRDA,Copper_USRDA,Magnesium_USRDA,Phosphorus_USRDA,Selenium_USRDA,Zinc_USRDA
0,16116,Legumes and Legume Products,"Soy flour, full-fat, roasted",441.0,34.80,21.86,33.67,7.61,9.7,6.0,...,0.5675,0.205375,0.723846,0.343333,0.156667,0.002468,0.878571,0.680000,0.136364,0.325455
1,18316,Baked Products,"Pie, coconut custard, commercially prepared",260.0,5.90,13.20,30.20,0.00,1.8,26.0,...,0.0475,0.025188,0.113846,0.073333,0.067500,0.000070,0.042857,0.174286,0.116364,0.061818
2,15261,Finfish and Shellfish Products,"Fish, tilapia, raw",96.0,20.08,1.70,0.00,0.00,0.0,0.0,...,0.0600,0.243938,0.048462,0.034167,0.008333,0.000083,0.064286,0.242857,0.760000,0.030000
3,8417,Breakfast Cereals,"Cereals, QUAKER, Instant Oatmeal, Banana Bread...",368.0,8.97,4.85,75.70,29.45,6.7,0.0,...,0.0000,0.706875,0.769231,0.816667,0.230833,0.000000,0.219048,0.450000,0.000000,0.188182
4,20022,Cereal Grains and Pasta,"Cornmeal, degermed, enriched, yellow",370.0,7.11,1.75,79.45,1.61,3.9,11.0,...,0.8375,0.310500,0.293846,0.459167,0.002500,0.000084,0.076190,0.141429,0.190909,0.060000


In [39]:
NAME_COLUMN = "FoodGroup"
df = df.rename(columns={NAME_COLUMN: "Nom"})


# Fonction moteur de recherche

def search_data(query, df, cols):
    """
    Recherche simple : 
    - l'utilisateur tape une requête ("Cereal", "Wheat", etc.)
    - on cherche chaque mot dans plusieurs colonnes
    - AND entre les mots, OR entre les colonnes
    """
    
    mots = query.lower().split()
    
    # Pour chaque mot, on construit un masque qui vérifie "mot présent dans AU MOINS une colonne"
    mask_mots = []
    
    for m in mots:
        mask_col = False
        for col in cols:
            # Vérifier que la colonne existe
            if col in df.columns:
                mask_col = mask_col | df[col].astype(str).str.lower().str.contains(m, na=False)
        mask_mots.append(mask_col)
    
    # AND entre les masques de chaque mot
    mask_final = mask_mots[0]
    for mask in mask_mots[1:]:
        mask_final = mask_final & mask
    
    return df[mask_final]


# Test

cols = ["Nom", "Descrip"]
test_query = "Cereal chocolate"
results = search_data(test_query, df, cols)

print(f"Résultats pour la recherche : '{test_query}'")
print("Nombre de lignes trouvées :", len(results))
display(results.head())

Résultats pour la recherche : 'Cereal chocolate'
Nombre de lignes trouvées : 11


,ID,Nom,Descrip,Energy_kcal,Protein_g,Fat_g,Carb_g,Sugar_g,Fiber_g,VitA_mcg,...,Folate_USRDA,Niacin_USRDA,Riboflavin_USRDA,Thiamin_USRDA,Calcium_USRDA,Copper_USRDA,Magnesium_USRDA,Phosphorus_USRDA,Selenium_USRDA,Zinc_USRDA
574,8631,Breakfast Cereals,"Cereals ready-to-eat, KELLOGG'S FROSTED MINI-W...",354.0,8.50,3.4,81.40,22.29,10.9,0.0,...,1.4850,0.555625,0.576923,0.558333,0.027500,0.0,0.214286,0.478571,0.0,0.245455
2942,8512,Breakfast Cereals,"Cereals, MALT-O-MEAL, chocolate, prepared with...",47.0,1.37,0.1,9.22,2.17,0.4,0.0,...,0.0000,0.165625,0.115385,0.125000,0.040833,0.0,0.009524,0.022857,0.0,0.018182
3487,8594,Breakfast Cereals,"Cereals ready-to-eat, GENERAL MILLS, Chocolate...",412.0,4.80,8.4,81.10,26.20,2.0,0.0,...,0.0000,1.043750,1.076923,1.083333,0.277500,0.0,0.064286,0.190000,0.0,1.136364
4043,8658,Breakfast Cereals,"Cereals ready-to-eat, KELLOGG'S KRAVE double c...",397.0,7.10,11.2,75.90,36.30,10.6,0.0,...,1.1025,1.043750,0.869231,1.041667,0.055833,0.0,0.114286,0.160000,0.0,0.454545
4197,8669,Breakfast Cereals,"Cereals ready-to-eat, GENERAL MILLS, FIBER ONE...",253.0,3.29,2.9,84.00,17.29,35.2,0.0,...,0.0000,1.043750,1.069231,1.075000,1.110833,0.0,0.126190,0.381429,0.0,1.136364


### Appel à Ollama : résumé des résultats filtrés

Cette cellule met en place une fonction `ask_ollama()` qui permet 
d'appeler un modèle hébergé localement via Ollama.

Objectifs :
- envoyer un prompt (texte) au modèle
- récupérer sa réponse
- tester avec un résumé simple du sous-ensemble filtré

Ollama doit être lancé en arrière-plan (via `ollama serve`).


In [51]:
# Fonction pour interroger Ollama

def ask_ollama(prompt, model="llama3"):
    """
    Appelle Ollama et retourne la réponse complète.
    Gère le streaming JSON ligne par ligne (format Ollama).
    """
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": model, "prompt": prompt},
        stream=True
    )

    response.raise_for_status()

    full_text = ""

    # Lecture streaming (Ollama envoie plusieurs JSON consécutifs)
    for line in response.iter_lines():
        if not line:
            continue
        try:
            part = json.loads(line.decode("utf-8"))
            full_text += part.get("response", "")
        except Exception:
            # Ignore les fragments mal formés
            continue

    return full_text

# Test : Résumé intelligent des données filtrées

subset = results.head(11)
subset_md = subset.to_markdown(index=False)

prompt_test = f"""
Voici un tableau d'aliments et leurs valeurs nutritionnelles :

{subset_md}

Tâche :
1. Décris en 4 à 6 phrases le type d'aliments visibles.
2. Résume les tendances nutritionnelles (aliments énergétiques, protéiques, gras...).

Reste général : ne cite pas de valeurs numériques exactes.

Réponse :
"""

llm_output = ask_ollama(prompt_test, model="llama3")
print(llm_output)


Type d'aliments :

Les aliments présentés dans ce tableau sont des céréales pour petit déjeuner prêtes à être consommées, notamment des céréales Kellogg's et General Mills. Il s'agit principalement de céréales sucrées et enrichies en vitamines et minéraux.

Tendances nutritionnelles :

Les céréales présentées ont tendance à être énergétiques, avec un apport calorique moyen supérieur à 350 kcal par portion. Les protéines sont généralement présentes dans des quantités modérées, entre 3 et 8 grammes par portion. Les graisses sont également présentes, mais en moindre quantité que les protéines, avec un apport moyen de 2-5 grammes par portion. En ce qui concerne les glucides, ils sont généralement les principaux constituants des céréales, avec un apport moyen supérieur à 70 grammes par portion.

Général :

Les céréales pour petit déjeuner présentées ont tendance à être sucrées et énergétiques, avec des quantités variables de protéines, graisses et glucides. Ces aliments sont destinés à four

In [50]:
# Génération automatique de questions discriminantes

def generate_questions(df_subset, model="llama3"):

    subset_md = df_subset.to_markdown(index=False)

    prompt = f"""
Voici un extrait d'une base de données sous forme de tableau :

{subset_md}

Objectif :
Proposer 5 à 10 questions pertinentes qu'un utilisateur pourrait facilement comprendre, qui permettent de réduire l'incertitude
sur le profil nutritionnel d'un aliment.

Contraintes :
- Les questions doivent être courtes.
- Elles doivent permettre de comprendre les principales variables chimiques :
  protéines, fibres, énergie ...
- Ne fait pas de résumé : uniquement des questions.

Réponds en français, sous forme de liste numérotée.
"""

    return ask_ollama(prompt, model=model)


# Test de génération de questions
questions_output = generate_questions(subset, model="llama3")
print(questions_output)


Voici les 10 questions pertinentes pour comprendre le profil nutritionnel d'un aliment :

1. Quel est le taux de protéines dans cet aliments ?
2. Est-ce que l'aliment contient des fibres et si oui, en combien ?
3. Combien d'énergie (en kcal) offre cet aliment par portion ?
4. Le produit contient-il du sucre ? Si oui, combien ?
5. Quel est le ratio entre les glucides et les protéines dans cet aliments ?
6. Est-ce que l'aliment contient des graisses saturées ou insaturées ?
7. Combien de calories sont apportées par 100 grammes d'aliments ?
8. Y a-t-il des ingrédients qui pourraient causer une allergie ou une intolérance ?
9. Le produit est-il enrichi en vitamines ou minéraux particuliers (par exemple, calcium, fer) ?
10. Est-ce que l'aliment est considéré comme un aliments riche en glucides simples (sucre simple) ou complexes (fibres) ?
